# LeetCode 3791 — Number of Balanced Integers in a Range bằng Python

- Link: https://leetcode.com/problems/number-of-balanced-integers-in-a-range/
- Độ khó: **Hard**
- Chủ đề: Digit DP, tổng đan dấu của chữ số, kỹ thuật `f(high) − f(low − 1)`, đếm tổ hợp bằng phân phối tổng chữ số (tích chập)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countBalanced(self, low: int, high: int) -> int:
```

## 1. Mục tiêu học

1. Viết điều kiện "tổng chữ số ở vị trí lẻ = tổng ở vị trí chẵn" thành **tổng đan dấu** `d₁ − d₂ + d₃ − … = 0`.
2. Nhận ra: thêm số 0 ở đầu **đảo dấu** toàn bộ tổng đan dấu (nếu thêm lẻ số 0) — nhưng điều kiện "bằng 0" **không đổi** ⇒ đệm số 0 trong digit DP là an toàn.
3. Viết **Digit DP** với trạng thái `(vị trí, tổng đan dấu hiện tại, chặn trên)`.
4. Đếm bằng **tổ hợp**: phần tự do sau vị trí rẽ đếm bằng tích chập phân phối tổng chữ số.
5. Xử lý điều kiện "ít nhất hai chữ số".

## 2. Đề bài & ví dụ

**Phát biểu lại.** Số nguyên **cân bằng** nếu: có ít nhất hai chữ số, và tổng các chữ số ở vị trí chẵn bằng tổng ở vị trí lẻ (vị trí đánh số từ 1 tính từ trái). Đếm số cân bằng trong `[low, high]`.

**Ví dụ 1:** `low = 1, high = 100` → `9` (`11, 22, …, 99`).

**Ví dụ 2:** `low = 120, high = 129` → `1` (`121`).

**Ví dụ 3:** `low = high = 1234` → `0` (`1 + 3 ≠ 2 + 4`).

**Ràng buộc:** `1 ≤ low ≤ high ≤ 10^{15}`.

In [1]:
from functools import cache
import random, time

examples = [(1, 100, 9), (120, 129, 1), (1234, 1234, 0)]
for lo, hi, exp in examples:
    print(lo, hi, "-> kỳ vọng", exp)

1 100 -> kỳ vọng 9
120 129 -> kỳ vọng 1
1234 1234 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Tổng đan dấu và cân bằng

In [2]:
def alt_sum(x):
    s = str(x)
    return sum(int(ch) * (1 if i % 2 == 0 else -1) for i, ch in enumerate(s))

def is_balanced(x):
    return x >= 10 and alt_sum(x) == 0

print([x for x in range(1, 200) if is_balanced(x)])
print(alt_sum(1234), alt_sum(121))

[11, 22, 33, 44, 55, 66, 77, 88, 99, 110, 121, 132, 143, 154, 165, 176, 187, 198]
-2 0


### 3.2. Đệm số 0 đầu không làm hỏng điều kiện

Với digit DP trên chuỗi `N` đã cố định độ dài, các số ngắn hơn được coi như có số 0 đầu. Thêm `t` số 0 đầu dịch mọi chữ số đi `t` vị trí: nếu `t` lẻ, mọi dấu đảo ⇒ tổng đan dấu thành `−S`. Điều kiện `S = 0` ⇔ `−S = 0` ⇒ **không đổi**.

In [3]:
for x in (121, 1210, 11, 2002):
    s = str(x)
    for pad in range(0, 3):
        t = "0" * pad + s
        v = sum(int(ch) * (1 if i % 2 == 0 else -1) for i, ch in enumerate(t))
        print(f"{t:>6}: tổng đan dấu {v:3}", end=" | ")
    print()

   121: tổng đan dấu   0 |   0121: tổng đan dấu   0 |  00121: tổng đan dấu   0 | 
  1210: tổng đan dấu   0 |  01210: tổng đan dấu   0 | 001210: tổng đan dấu   0 | 
    11: tổng đan dấu   0 |    011: tổng đan dấu   0 |   0011: tổng đan dấu   0 | 
  2002: tổng đan dấu   0 |  02002: tổng đan dấu   0 | 002002: tổng đan dấu   0 | 


### 3.3. Phân phối tổng của `k` chữ số

`cnt_k[s]` = số cách chọn `k` chữ số (0–9) có tổng `s`. Tính bằng DP/tích chập: `cnt_{k+1}[s] = Σ_{d=0}^{9} cnt_k[s − d]`.

In [4]:
def digit_sum_dist(k):
    dist = [1]
    for _ in range(k):
        nd = [0] * (len(dist) + 9)
        for s, c in enumerate(dist):
            for dgt in range(10):
                nd[s + dgt] += c
        dist = nd
    return dist

print(digit_sum_dist(2)[:12], "| tổng =", sum(digit_sum_dist(2)))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 9, 8] | tổng = 100


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`121`: vị trí 1 và 3 (lẻ) là `1, 1` tổng 2; vị trí 2 (chẵn) là `2` ⇒ cân bằng. Tổng đan dấu `1 − 2 + 1 = 0`.

**Hai chữ số:** `ab` cân bằng ⇔ `a = b` ⇒ `11, 22, …, 99` (9 số) — đúng ví dụ 1.

**Điều kiện "ít nhất hai chữ số"** chỉ loại các số một chữ số; nhưng một chữ số `d ≥ 1` có tổng đan dấu `d ≠ 0` nên vốn không cân bằng. Chỉ số `0` có tổng 0 — và `0` không nằm trong `[low, high]` vì `low ≥ 1`. Khi đếm `[0, N]` bằng digit DP, nhớ **trừ số 0**.

Vì `high` tới `10^{15}`, dùng `f(high) − f(low − 1)` với `f(N)` = số cân bằng trong `[1, N]`.

## 5. Cách 1 — Brute Force: duyệt từng số

In [5]:
def brute(low, high):
    return sum(1 for x in range(low, high + 1) if is_balanced(x))

for lo, hi, exp in examples:
    print(brute(lo, hi), "kỳ vọng", exp)

9 kỳ vọng 9
1 kỳ vọng 1
0 kỳ vọng 0


**Nút thắt:** đoạn tới `10^{15}`. Cần đếm theo cấu trúc chữ số.

## 6. Ý tưởng thuật toán / bước đột phá

**Digit DP** trên chuỗi `N` (không cần đệm thêm, nhờ mục 3.2):

- trạng thái `(pos, diff, tight)`, `diff` = tổng đan dấu của tiền tố (dấu theo vị trí trong chuỗi `N`);
- tại `pos`, chữ số `d ∈ [0, hi]`, `diff += d` nếu `pos` chẵn (0-indexed), ngược lại `diff −= d`;
- cuối cùng đếm 1 nếu `diff == 0`.

Kết quả đếm cả số `0` (tổng 0) ⇒ trừ 1. `diff ∈ [−72, 72]` (tối đa 8 chữ số mỗi dấu với 16 chữ số) ⇒ số trạng thái nhỏ.

**Cách tổ hợp:** đi dọc chuỗi `N`; khi rẽ chữ số nhỏ hơn tại `pos`, phần còn lại gồm `p` vị trí dấu `+` và `q` vị trí dấu `−`, cần tổng `+ − tổng − = −diff`. Số cách = `Σ_s cnt_p[s] · cnt_q[s + diff]` (tích chập hai phân phối).

## 7. Các cách tiếp theo

- **Cách 2:** Digit DP đệ quy có nhớ (lời giải nộp chính).
- **Cách 3:** đi dọc đường chặn + đếm phần tự do bằng **tích chập phân phối tổng chữ số**.
- **Cách 4:** Digit DP **dạng bảng**: `dict diff → số cách` cho các tiền tố "đã nhỏ hơn", cập nhật từng chữ số.

### 7.1. Cách 2 — Digit DP đệ quy (lời giải chính)

In [6]:
def digit_dp(low, high):
    def f(N):
        if N <= 0:
            return 0
        s = str(N)

        @cache
        def dfs(pos, diff, tight):
            if pos == len(s):
                return 1 if diff == 0 else 0
            hi = int(s[pos]) if tight else 9
            sign = 1 if pos % 2 == 0 else -1
            return sum(dfs(pos + 1, diff + sign * d, tight and d == hi) for d in range(hi + 1))

        return dfs(0, 0, True) - 1            # trừ số 0
    return f(high) - f(low - 1)

for lo, hi, exp in examples:
    print(digit_dp(lo, hi), "kỳ vọng", exp)

9 kỳ vọng 9
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp:** trạng thái `16 · 145 · 2`, mỗi trạng thái 10 nhánh ⇒ ~`5·10^4` phép mỗi `f`.

### 7.2. Cách 3 — Đường chặn + tích chập

In [7]:
DIST = [digit_sum_dist(k) for k in range(17)]

def free_ways(p, q, need):
    """số cách điền p chữ số dấu + và q chữ số dấu − sao cho (tổng+) − (tổng−) = need."""
    A, B = DIST[p], DIST[q]
    total = 0
    for s, c in enumerate(A):
        t = s - need
        if 0 <= t < len(B):
            total += c * B[t]
    return total

def combinatorial(low, high):
    def f(N):
        if N <= 0:
            return 0
        s = str(N); L = len(s); total = 0; diff = 0
        for pos in range(L):
            cur = int(s[pos]); sign = 1 if pos % 2 == 0 else -1
            rest = range(pos + 1, L)
            p = sum(1 for i in rest if i % 2 == 0); q = len(rest) - p
            for d in range(cur):
                total += free_ways(p, q, -(diff + sign * d))
            diff += sign * cur
        total += diff == 0                    # chính N
        return total - 1                      # trừ số 0
    return f(high) - f(low - 1)

for lo, hi, exp in examples:
    print(combinatorial(lo, hi), "kỳ vọng", exp)

9 kỳ vọng 9
1 kỳ vọng 1
0 kỳ vọng 0


**Độ phức tạp:** `16 · 10` lần tích chập, mỗi lần `O(73)` ⇒ ~`10^4`.

### 7.3. Cách 4 — Digit DP dạng bảng

In [8]:
from collections import defaultdict
def tabulated(low, high):
    def f(N):
        if N <= 0:
            return 0
        s = str(N)
        free = defaultdict(int)           # diff -> số tiền tố đã nhỏ hơn N
        tight_diff = 0
        for pos, ch in enumerate(s):
            cur = int(ch); sign = 1 if pos % 2 == 0 else -1
            nf = defaultdict(int)
            for dv, c in free.items():
                for d in range(10):
                    nf[dv + sign * d] += c
            for d in range(cur):
                nf[tight_diff + sign * d] += 1
            tight_diff += sign * cur
            free = nf
        return free[0] + (tight_diff == 0) - 1
    return f(high) - f(low - 1)

for lo, hi, exp in examples:
    print(tabulated(lo, hi), "kỳ vọng", exp)

9 kỳ vọng 9
1 kỳ vọng 1
0 kỳ vọng 0


In [9]:
class SolutionTab:
    def countBalanced(self, low: int, high: int) -> int:
        return tabulated(low, high)

print([SolutionTab().countBalanced(lo, hi) for lo, hi, _ in examples])

[9, 1, 0]


**Độ phức tạp Cách 4:** mỗi vị trí `O(145 · 10)` ⇒ ~`2·10^4`.

## 8. TRACE từng bước

Trace Cách 3 với `N = 129` (đếm số cân bằng trong `[1, 129]`).

In [10]:
def trace(N):
    s = str(N); L = len(s); total = 0; diff = 0
    for pos in range(L):
        cur = int(s[pos]); sign = 1 if pos % 2 == 0 else -1
        rest = range(pos + 1, L); p = sum(1 for i in rest if i % 2 == 0); q = len(rest) - p
        add = sum(free_ways(p, q, -(diff + sign * d)) for d in range(cur))
        total += add
        print(f"pos={pos} chữ số {cur} (dấu {'+' if sign > 0 else '-'}): rẽ d<{cur}, còn {p} vị trí '+' và {q} vị trí '-' -> cộng {add}, tổng={total}")
        diff += sign * cur
    total += diff == 0
    print(f"chính N có diff={diff} -> tổng={total}; trừ số 0 -> {total - 1}")
    return total - 1

trace(129)
print("vét cạn [1,129]:", brute(1, 129), "=", [x for x in range(1, 130) if is_balanced(x)])

pos=0 chữ số 1 (dấu +): rẽ d<1, còn 1 vị trí '+' và 1 vị trí '-' -> cộng 10, tổng=10
pos=1 chữ số 2 (dấu -): rẽ d<2, còn 1 vị trí '+' và 0 vị trí '-' -> cộng 1, tổng=11
pos=2 chữ số 9 (dấu +): rẽ d<9, còn 0 vị trí '+' và 0 vị trí '-' -> cộng 1, tổng=12
chính N có diff=8 -> tổng=12; trừ số 0 -> 11
vét cạn [1,129]: 11 = [11, 22, 33, 44, 55, 66, 77, 88, 99, 110, 121]


## 9. Code nộp LeetCode

In [11]:
from functools import cache

class Solution:
    def countBalanced(self, low: int, high: int) -> int:
        def f(N):
            if N <= 0:
                return 0
            s = str(N)

            @cache
            def dfs(pos, diff, tight):
                if pos == len(s):
                    return 1 if diff == 0 else 0
                hi = int(s[pos]) if tight else 9
                sign = 1 if pos % 2 == 0 else -1
                res = 0
                for d in range(hi + 1):
                    res += dfs(pos + 1, diff + sign * d, tight and d == hi)
                return res

            return dfs(0, 0, True) - 1           # bỏ số 0 (tổng đan dấu 0 nhưng chỉ một chữ số)

        return f(high) - f(low - 1)

sol = Solution()
for lo, hi, exp in examples:
    got = sol.countBalanced(lo, hi)
    print(lo, hi, "->", got, "✔" if got == exp else "✘")

1 100 -> 9 ✔
120 129 -> 1 ✔
1234 1234 -> 0 ✔


## 10. Vì sao thuật toán đúng

1. **Tương đương:** cân bằng ⇔ (≥ 2 chữ số) và tổng đan dấu `= 0`. Số một chữ số `d ≥ 1` có tổng đan dấu `d ≠ 0`; chỉ `0` có tổng 0 và bị trừ riêng.
2. **Đệm 0:** với số `x < N` có ít chữ số hơn, digit DP xem `x` với các số 0 đầu; theo mục 3.2, tổng đan dấu của chuỗi đệm bằng `±` tổng thật ⇒ điều kiện `= 0` giữ nguyên.
3. **Digit DP:** chuẩn — mỗi `x ∈ [0, N]` tương ứng đúng một đường đi trong cây quyết định chữ số với ràng buộc `tight`.
4. **Tổ hợp:** phân hoạch `x < N` theo vị trí rẽ đầu tiên; phần sau tự do, đếm bằng tích chập phân phối tổng của các vị trí `+` và `−`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt | `O((high − low) · D)` | `O(1)` |
| 2. Digit DP đệ quy | `O(D · S · 10)` với `S ≈ 145` | `O(D · S)` |
| 3. Tích chập | `O(D · 10 · S)` | `O(D · S)` |
| 4. DP dạng bảng | `O(D · S · 10)` | `O(S)` |

## 12. Bộ test

In [12]:
def run_all(lo, hi, with_brute=True):
    res = [digit_dp(lo, hi), combinatorial(lo, hi), tabulated(lo, hi), Solution().countBalanced(lo, hi)]
    if with_brute:
        res.append(brute(lo, hi))
    return res

for lo, hi, exp in examples:
    assert all(v == exp for v in run_all(lo, hi))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng duyệt)
cases = [(1, 9, 0), (1, 10, 0), (11, 11, 1), (1, 1000, None), (10**15, 10**15, 0), (909, 909, 0), (121, 121, 1)]
for lo, hi, exp in cases:
    r = run_all(lo, hi, with_brute=hi - lo <= 10**5)
    assert len(set(r)) == 1, (lo, hi, r)
    if exp is not None: assert r[0] == exp
    print((lo, hi), "->", r[0])

(1, 9) -> 0
(1, 10) -> 0
(11, 11) -> 1
(1, 1000) -> 54
(1000000000000000, 1000000000000000) -> 0
(909, 909) -> 0
(121, 121) -> 1


In [14]:
# 12.3 phản ví dụ: đếm số chia hết cho 11 (chỉ là điều kiện CẦN, không đủ)
div11 = [x for x in range(10, 1000) if x % 11 == 0 and not is_balanced(x)]
print("chia hết cho 11 nhưng không cân bằng:", div11[:8], "...")

chia hết cho 11 nhưng không cân bằng: [209, 308, 319, 407, 418, 429, 506, 517] ...


In [15]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3791)
for _ in range(150):
    lo = rng.randint(1, 10**rng.randint(1, 12)); hi = lo + rng.randint(0, 3000)
    r = run_all(lo, hi)
    assert len(set(r)) == 1, (lo, hi, r)
for _ in range(100):
    lo = rng.randint(1, 10**15); hi = rng.randint(lo, 10**15)
    r = run_all(lo, hi, with_brute=False)
    assert len(set(r)) == 1
print("250 test ngẫu nhiên: OK")

250 test ngẫu nhiên: OK


In [16]:
# 12.5 tốc độ
t0 = time.time()
for _ in range(100): Solution().countBalanced(1, 10**15)
print("100 lần [1, 10^15]:", round(time.time() - t0, 2), "s | đáp án:", Solution().countBalanced(1, 10**15))

100 lần [1, 10^15]: 0.16 s | đáp án: 32811494188974


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ số 0

In [17]:
def no_minus_zero(N):
    s = str(N)
    @cache
    def dfs(pos, diff, tight):
        if pos == len(s): return int(diff == 0)
        hi = int(s[pos]) if tight else 9; sign = 1 if pos % 2 == 0 else -1
        return sum(dfs(pos + 1, diff + sign * d, tight and d == hi) for d in range(hi + 1))
    return dfs(0, 0, True)                         # SAI: gồm cả 0

print("f(100) sai:", no_minus_zero(100), "| đúng:", Solution().countBalanced(1, 100))

f(100) sai: 10 | đúng: 9


### Lỗi 2 — Đánh dấu vị trí theo chữ số **từ phải** nhưng trộn với chuỗi đệm

Không sai nếu nhất quán (vẫn chỉ đảo dấu). Lỗi thật sự là **đổi quy ước giữa chừng**, ví dụ dấu theo vị trí trong chuỗi `N` nhưng chữ số của `x` được tính theo vị trí trong chính `x` (không đệm) — trộn hai hệ quy chiếu.

In [18]:
def wrong_mixed(N):
    # SAI: dùng digit DP nhưng ở lá lại kiểm tra theo độ dài thực của x (không biết x nên giả định độ dài N)
    s = str(N); cnt = 0
    for x in range(1, N + 1):
        t = str(x)
        signs = [1 if (i + (len(s) - len(t))) % 2 == 0 else -1 for i in range(len(t))]
        v = sum(int(ch) * sg for ch, sg in zip(t, signs))
        cnt += v == 0 and x >= 10
    return cnt

print("hệ quy chiếu đệm cho cùng kết quả:", wrong_mixed(500) == Solution().countBalanced(1, 500), "(minh họa: đảo dấu không ảnh hưởng)")

hệ quy chiếu đệm cho cùng kết quả: True (minh họa: đảo dấu không ảnh hưởng)


### Lỗi 3 — Không xóa cache giữa `f(high)` và `f(low − 1)`

Nếu dùng `@cache` ở cấp ngoài với chuỗi `s` là biến toàn cục, cache của lần gọi đầu bị dùng sai cho lần sau.

In [19]:
STATE = {"s": ""}
@cache
def shared(pos, diff, tight):
    s = STATE["s"]
    if pos == len(s): return int(diff == 0)
    hi = int(s[pos]) if tight else 9; sign = 1 if pos % 2 == 0 else -1
    return sum(shared(pos + 1, diff + sign * d, tight and d == hi) for d in range(hi + 1))

STATE["s"] = "200"; a = shared(0, 0, True) - 1
STATE["s"] = "129"; b = shared(0, 0, True) - 1            # SAI: cache cũ của "200"
print("sai:", a - b, "| đúng:", Solution().countBalanced(130, 200))

sai: 0 | đúng: 7


### Lỗi 4 — Coi "cân bằng" ⇔ "chia hết cho 11"

Tổng đan dấu `≡ x (mod 11)`, nên cân bằng ⇒ chia hết cho 11, nhưng không ngược lại (ví dụ `209`: `2 − 0 + 9 = 11`). Xem 12.3.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Điều kiện trên chữ số + đoạn tới `10^{15}–10^{18}`** ⇒ digit DP, `f(high) − f(low − 1)`.
- **Tổng có dấu theo vị trí** ⇒ trạng thái `diff` trong khoảng nhỏ; đệm 0 chỉ đảo dấu.
- Nhớ các phần tử "biên": số 0, số một chữ số.
- Liên quan: LC 2999 (đếm số "mạnh"), LC 2801 (stepping numbers), LC 1012.

## 15. Biến thể của bài

1. **Tổng vị trí lẻ lớn hơn tổng vị trí chẵn**: đếm `diff > 0` (khi đó đệm 0 **có** ảnh hưởng vì dấu đảo — phải đánh vị trí theo độ dài thật, cần cờ `started`).
2. **Cân bằng theo từng nửa** (tổng nửa đầu = nửa sau): trạng thái phụ thuộc độ dài thật.
3. **Chia hết cho 11**: chỉ cần `diff mod 11`.
4. **Đếm số cân bằng có đúng `k` chữ số**: `free_ways` trực tiếp.

In [20]:
# Biến thể 4: số cân bằng có đúng k chữ số (chữ số đầu khác 0)
def exact_k(k):
    p = (k + 1) // 2; q = k // 2                   # số vị trí '+' và '-'
    total = 0
    for first in range(1, 10):                     # vị trí 1 mang dấu +
        total += free_ways(p - 1, q, -first)
    return total

for k in range(2, 6):
    assert exact_k(k) == sum(1 for x in range(10 ** (k - 1), 10 ** k) if is_balanced(x))
print([exact_k(k) for k in range(2, 10)])

[9, 45, 615, 4170, 50412, 381723, 4379055, 35235465]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt | Kiểm tra từng số | `O(Δ D)` | `O(1)` | Kiểm thử |
| 2. Digit DP đệ quy | `(pos, diff, tight)` | ~`5·10^4` | `O(D S)` | **Lời giải chuẩn** |
| 3. Tích chập | Đuôi tự do theo phân phối | ~`10^4` | `O(D S)` | Khi cần đếm nhiều lần |
| 4. DP bảng | `dict` diff | ~`2·10^4` | `O(S)` | Không đệ quy |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countBalanced(100, 200)` = ?

**Bài 2 (tính tay).** Có bao nhiêu số cân bằng có ba chữ số? (Gợi ý: `a − b + c = 0` ⇔ `b = a + c`.)

**Bài 3 (điền TODO).** Viết `count_div11(low, high)` bằng digit DP với `diff mod 11`.

**Bài 4 (tự viết lại).** Viết lại Cách 2 với lời giải đếm **số không cân bằng** và kiểm tra tổng = độ dài đoạn.

**Bài 5 (mini challenge).** Biến thể 1: đếm số (≥ 2 chữ số) có tổng vị trí lẻ **lớn hơn** tổng vị trí chẵn.

In [21]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_div11(low, high):
    # TODO: digit DP trạng thái (pos, diff % 11, tight)
    pass

In [22]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_odd_greater(low, high):
    # TODO: cần cờ started để đánh vị trí theo độ dài thật
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dạng `1bc` với `b = 1 + c`. (2) `c = 0..8` ⇒ 9 số `110, 121, …, 198`. (3) `200`: `2 − 0 + 0 ≠ 0` ⇒ đáp án `9`.

**Bài 2.** (1) `a ∈ 1..9`, `c ∈ 0..9`, `b = a + c ≤ 9`. (2) Với mỗi `a` có `10 − a` lựa chọn `c`. (3) `Σ_{a=1}^{9} (10 − a) = 45`.

**Bài 3.** (1) `diff` lấy mod 11. (2) Kết thúc khi `diff % 11 == 0`. (3) Trừ số 0; đối chiếu `x % 11 == 0`.

**Bài 4.** (1) Không cân bằng = độ dài đoạn − cân bằng. (2) Đếm trực tiếp `diff ≠ 0` cũng được. (3) So tổng.

**Bài 5.** (1) Trạng thái thêm `started` và `len_parity`… hoặc tính vị trí từ chữ số đầu khác 0. (2) Đơn giản hơn: digit DP cho từng độ dài `k` riêng (`k` chữ số, chữ số đầu ≠ 0). (3) Cộng cho mọi độ dài và kiểm tra bằng duyệt.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().countBalanced(100, 200) == 9 == brute(100, 200)
assert Solution().countBalanced(100, 999) == 45 == exact_k(3)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [24]:
# Bài 3
def count_div11_sol(low, high):
    def f(N):
        if N <= 0: return 0
        s = str(N)
        @cache
        def dfs(pos, r, tight):
            if pos == len(s): return int(r == 0)
            hi = int(s[pos]) if tight else 9; sign = 1 if pos % 2 == 0 else -1
            return sum(dfs(pos + 1, (r + sign * d) % 11, tight and d == hi) for d in range(hi + 1))
        return dfs(0, 0, True) - 1
    return f(high) - f(low - 1)

assert count_div11_sol(1, 1000) == sum(1 for x in range(1, 1001) if x % 11 == 0)
assert count_div11_sol(12345, 99999) == sum(1 for x in range(12345, 100000) if x % 11 == 0)
print("Bài 3 OK")

Bài 3 OK


In [25]:
# Bài 4
def count_unbalanced(low, high):
    def f(N):
        if N <= 0: return 0
        s = str(N)
        @cache
        def dfs(pos, diff, tight):
            if pos == len(s): return int(diff != 0)
            hi = int(s[pos]) if tight else 9; sign = 1 if pos % 2 == 0 else -1
            return sum(dfs(pos + 1, diff + sign * d, tight and d == hi) for d in range(hi + 1))
        return dfs(0, 0, True)                     # số 0 có diff 0 nên tự không bị đếm
    return f(high) - f(low - 1)

rng = random.Random(4)
for _ in range(100):
    lo = rng.randint(1, 10**14); hi = rng.randint(lo, 10**15)
    assert count_unbalanced(lo, hi) + Solution().countBalanced(lo, hi) == hi - lo + 1
print("Bài 4 OK")

Bài 4 OK


In [26]:
# Bài 5: tổng vị trí lẻ > tổng vị trí chẵn (vị trí tính theo độ dài thật)
def count_odd_greater_sol(low, high):
    def count_len_upto(k, N):
        # số k chữ số (chữ số đầu khác 0) <= N có alt_sum > 0
        s = str(N)
        if len(s) < k: return 0
        if len(s) > k: s = "9" * k
        @cache
        def dfs(pos, diff, tight):
            if pos == k: return int(diff > 0)
            lo_d = 1 if pos == 0 else 0
            hi = int(s[pos]) if tight else 9; sign = 1 if pos % 2 == 0 else -1
            return sum(dfs(pos + 1, diff + sign * d, tight and d == hi) for d in range(lo_d, hi + 1))
        return dfs(0, 0, True)
    def f(N):
        if N < 10: return 0
        return sum(count_len_upto(k, N) for k in range(2, len(str(N)) + 1))
    return f(high) - f(low - 1)

def brute_odd_greater(lo, hi):
    return sum(1 for x in range(max(lo, 10), hi + 1) if alt_sum(x) > 0)

rng = random.Random(5)
for _ in range(60):
    lo = rng.randint(1, 10**rng.randint(1, 6)); hi = lo + rng.randint(0, 2000)
    assert count_odd_greater_sol(lo, hi) == brute_odd_greater(lo, hi), (lo, hi)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def f(N):                       # số cân bằng trong [1, N]
    s = str(N)
    @cache
    def dfs(pos, diff, tight):
        if pos == len(s): return diff == 0
        hi = int(s[pos]) if tight else 9
        sign = 1 if pos % 2 == 0 else -1
        return sum(dfs(pos+1, diff + sign*d, tight and d == hi) for d in range(hi+1))
    return dfs(0, 0, True) - 1  # bỏ số 0
return f(high) - f(low - 1)
```

**Checklist:**

- [ ] Trừ số 0 (tổng đan dấu 0)?
- [ ] Số một chữ số tự động không cân bằng (trừ 0)?
- [ ] Hiểu vì sao đệm số 0 an toàn cho điều kiện `= 0` (nhưng **không** an toàn cho `> 0`)?
- [ ] Cache tách biệt giữa hai lần gọi `f`?
- [ ] `f(0) = 0` khi `low = 1`?